# Batch Runner — Google Colab (via GitHub)

## Prerequis AVANT d'executer (a faire une seule fois)

### 1. Pousser le code sur GitHub (sur ton PC)
```bash
git add .
git commit -m "batch runner colab"
git push
```

### 2. Creer un GitHub Personal Access Token
- Va sur github.com → **Settings** → **Developer settings** → **Personal access tokens** → **Tokens (classic)**
- Clique **Generate new token**
- Coche le scope `repo`
- Copie le token genere (commence par `ghp_...`)

### 3. Ajouter les secrets dans Colab
Icone **cle** dans le panneau gauche → activer l'acces pour chaque secret :
```
GITHUB_TOKEN        → ghp_xxxxx (ton Personal Access Token)
KONG_CHAT_URL       → valeur dans ton .env
KONG_EMBED_URL      → valeur dans ton .env
KONG_API_KEY        → valeur dans ton .env
PINECONE_API_KEY    → valeur dans ton .env
PINECONE_HOST       → valeur dans ton .env
SHEET_ID            → valeur dans ton .env
SHEET_WORKSHEET     → ex: Langgraph_G3.5_2606
```

### 4. Executer les cellules dans l'ordre

## Cellule 1 — Cloner le repo GitHub

In [13]:
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO         = 'willy-devo/ProjetLangGraph_Willy'
LOCAL_ROOT   = '/content/ProjetLangGraph_Willy'

if os.path.exists(LOCAL_ROOT):
    print('Repo deja present — pull pour mettre a jour')
    os.system(f'git -C {LOCAL_ROOT} pull')
else:
    print('Clonage du repo...')
    ret = os.system(f'git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git {LOCAL_ROOT}')
    if ret != 0:
        raise RuntimeError('Clonage echoue — verifie GITHUB_TOKEN et le nom du repo')

os.chdir(LOCAL_ROOT)
print(f'Repertoire courant : {os.getcwd()}')

Repo deja present — pull pour mettre a jour
Repertoire courant : /content/ProjetLangGraph_Willy


## Cellule 2 — Installer les dependances

In [7]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--quiet'])
print('Package agentic4api installe')

Package agentic4api installe


## Cellule 3 — Charger les secrets et configurer l'environnement

In [8]:
import os
from google.colab import userdata

SECRETS = [
    'KONG_CHAT_URL', 'KONG_EMBED_URL', 'KONG_API_KEY',
    'PINECONE_API_KEY', 'PINECONE_HOST',
    'SHEET_ID', 'SHEET_WORKSHEET',
]

missing = []
for key in SECRETS:
    try:
        os.environ[key] = userdata.get(key)
        print(f'  OK       {key}')
    except Exception:
        missing.append(key)
        print(f'  MANQUANT {key}')

if missing:
    raise EnvironmentError(f'Ajoute ces secrets dans le panneau cle Colab : {missing}')

os.environ.setdefault('GOOGLE_AUTH_MODE',  'oauth2')
os.environ.setdefault('GEMINI_MODEL',      'gemini-3.5-flash')
os.environ.setdefault('TOOL_MODE',         'text')
os.environ.setdefault('DEBUG_MODE',        'false')
os.environ.setdefault('BATCH_WAIT_S',      '1')
os.environ.setdefault('MAX_OUTPUT_TOKENS', '1024')
os.environ.setdefault('TOP_K',             '10')
os.environ.setdefault('KONG_VERIFY_SSL',   'false')

print('\nEnvironnement configure')

  OK       KONG_CHAT_URL
  OK       KONG_EMBED_URL
  OK       KONG_API_KEY
  OK       PINECONE_API_KEY
  OK       PINECONE_HOST
  OK       SHEET_ID
  OK       SHEET_WORKSHEET

Environnement configure


## Cellule 4 — Authentification Google Sheets

/content/ProjetLangGraph_Willy/src


In [18]:
import sys
sys.path.insert(0, '/content/ProjetLangGraph_Willy/src')
print(sys.path[0])

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

# Remplace _client() de sheet_writer par l'auth Colab
# (pas besoin de token.json ni credentials.json)
import agentic4api.batch.sheet_writer as _sw
_sw._client = lambda: gspread.authorize(creds)

print('Auth Google Sheets OK')

/content/ProjetLangGraph_Willy/src
Auth Google Sheets OK


## Cellule 5 — Verifier le golden dataset

In [ ]:
import os
os.makedirs('logs', exist_ok=True)

from agentic4api.batch.golden import load_golden
golden = load_golden()
print(f'Golden dataset  : {len(golden)} questions')
print(f'Worksheet cible : {os.environ["SHEET_WORKSHEET"]}')

## Cellule 6 — Lancer le batch

Commence avec `LIMIT = 5` pour verifier que tout fonctionne,  
puis mets `LIMIT = None` pour lancer les 464 questions.

In [ ]:
import os
from agentic4api.batch.run_batch import run
from agentic4api.graph.build import build_graph

# ── Config du run ────────────────────────────────────────────
LIMIT = 5      # None = toutes les 464 questions
SKIP  = 0      # reprendre a partir de la question N (--skip)
# ─────────────────────────────────────────────────────────────

graph = build_graph(use_memory=False)

run(
    worksheet   = os.environ['SHEET_WORKSHEET'],
    limit       = LIMIT,
    skip        = SKIP,
    parallel    = False,
    batch_size  = 1,
    resume      = False,
    resume_from = None,
    graph       = graph,
)

## Cellule 7 — (Optionnel) Sauvegarder les logs sur Google Drive

Les logs JSONL sont perdus quand la session Colab se ferme.  
Execute cette cellule pour les sauvegarder sur Drive.

In [ ]:
import shutil, datetime
from google.colab import drive

drive.mount('/content/drive')

DEST = f'/content/drive/MyDrive/batch_logs_{datetime.date.today()}'
shutil.copytree('logs', DEST, dirs_exist_ok=True)
print(f'Logs sauvegardes dans Drive : {DEST}')